# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze a Croissant-based dataset using the `mlcroissant` library, referencing all data entities by their `@id`.

### Dataset Source
The dataset is described using a Croissant schema at:
<br>
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant
!pip install -q matplotlib

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
# For visualization
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Croissant Metadata object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
We will review the available record sets, fields, and their `@id` values using the dataset's Croissant schema.
Below, we print out each available RecordSet `@id`, its label, and the contained Field and Column `@id` entries.


In [ ]:
# List available RecordSets and their details by @id
record_sets = []

for rs in dataset.record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.label}")
    record_sets.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.label}, type: {field.data_type})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.id} (name: {col.label}, type: {col.data_type})")
    print()

## 3. Data Extraction
Extract data from each identified RecordSet using their `@id` fields and load into pandas DataFrames for further analysis.


In [ ]:
# We'll gather DataFrames keyed by RecordSet @id
dataframes = {}

for record_set_id in record_sets:
    # Each yielded record is keyed by Field or Column @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    print(df.columns.tolist())
    if len(df) > 0:
        display(df.head(3))

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate numeric filtering, outlier removal, normalization, and grouping.

_Please update the example below to your chosen RecordSet and fields using the printed `@id`s above._

In [ ]:
# For demonstration, select the first available RecordSet and identify a numeric column

# Replace these with the actual @id strings for your dataset
example_record_set_id = record_sets[0] if len(record_sets) > 0 else None
df = dataframes.get(example_record_set_id)
print(f"Using RecordSet: {example_record_set_id}")

# Try to automatically select a likely numeric field by heuristics
numeric_field_id = None
if df is not None and len(df) > 0:
    for col in df.columns:
        # Try to select a column with float or int dtype
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    print('No numeric field found in example RecordSet. Please update numeric_field_id to a valid @id.')
else:
    print(f"Using numeric field @id: {numeric_field_id}")

if numeric_field_id:
    # Threshold for filtering (example: above mean value)
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a non-numeric field for grouping
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped records by {group_field_id} with mean of {numeric_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization

Let's visualize the distribution of the selected numeric variable and the group means, using only `@id`-referenced fields.

In [ ]:
if numeric_field_id and df is not None and len(df) > 0:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=30, edgecolor='k')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If we created a grouped mean, plot it as bar chart
    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(10, 5))
        plt.bar(grouped_df[group_field_id].astype(str), grouped_df[numeric_field_id])
        plt.title(f'{numeric_field_id} mean by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load, inspect, and analyze a structured dataset described by a Croissant schema using `mlcroissant`. We referenced all RecordSets, Fields, and Columns using their `@id` fields, enabling robust and portable data exploration. For further analysis, you may extend the above workflow to apply more domain-specific data wrangling, statistical modeling, or visualization as needed.